# Day 073 — Exercise 4: synthesize_segments

**What you'll build:** `synthesize_segments(segments, tts_fn=None) -> list[bytes]` — synthesize a full script to a list of audio byte blobs.

**Why it matters:** `synthesize_segments` is the script-to-audio pipeline. It drives the PodcastGenerator and produces the ordered list of audio parts that become numbered output files.

In [ ]:
import asyncio
import asyncio
from pathlib import Path

COMMON_VOICES = [
    {'ShortName': 'en-US-AriaNeural',    'Gender': 'Female', 'Locale': 'en-US'},
    {'ShortName': 'en-US-GuyNeural',     'Gender': 'Male',   'Locale': 'en-US'},
    {'ShortName': 'en-GB-LibbyNeural',   'Gender': 'Female', 'Locale': 'en-GB'},
    {'ShortName': 'en-AU-NatashaNeural', 'Gender': 'Female', 'Locale': 'en-AU'},
    {'ShortName': 'fr-FR-DeniseNeural',  'Gender': 'Female', 'Locale': 'fr-FR'},
    {'ShortName': 'de-DE-KatjaNeural',   'Gender': 'Female', 'Locale': 'de-DE'},
]

DEFAULT_VOICE_MAP = {
    'host':     'en-US-AriaNeural',
    'guest':    'en-US-GuyNeural',
    'narrator': 'en-GB-LibbyNeural',
}

def build_prosody_ssml(text, rate='+0%', pitch='+0Hz', volume='+0%'):
    return (
        '<speak version="1.0" '
        'xmlns="http://www.w3.org/2001/10/synthesis" xml:lang="en-US">'
        f'<prosody rate="{rate}" pitch="{pitch}" volume="{volume}">'
        f'{text}'
        '</prosody></speak>'
    )

def select_voice(voices, locale='en-US', gender=None):
    for v in voices:
        if v.get('Locale') != locale:
            continue
        if gender is not None and v.get('Gender', '').lower() != gender.lower():
            continue
        return v
    return None

def synthesize(text, voice='en-US-AriaNeural', tts_fn=None, rate='+0%', pitch='+0Hz'):
    if tts_fn is not None:
        return tts_fn(text, voice=voice, rate=rate, pitch=pitch)
    import edge_tts
    async def _run():
        comm = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
        chunks = []
        async for chunk in comm.stream():
            if chunk['type'] == 'audio':
                chunks.append(chunk['data'])
        return b''.join(chunks)
    return asyncio.run(_run())

_mock_tts = lambda text, **kw: b'AUDIO:' + text[:12].encode()


## Task

Implement `synthesize_segments`:

```
out = []
for seg in segments:
    audio = synthesize(seg['text'],
                       voice=seg.get('voice', 'en-US-AriaNeural'),
                       tts_fn=tts_fn,
                       rate=seg.get('rate', '+0%'),
                       pitch=seg.get('pitch', '+0Hz'))
    out.append(audio)
return out
```

## Your Implementation

In [ ]:
def synthesize_segments(segments: list, tts_fn=None) -> list:
    """Synthesize a list of segment dicts to audio bytes.

    Each segment: {text, voice?, rate?, pitch?}
    Defaults: voice='en-US-AriaNeural', rate='+0%', pitch='+0Hz'
    Returns: list of bytes, one per segment
    """
    raise NotImplementedError


In [ ]:
def synthesize_segments(segments, tts_fn=None):
    out = []
    for seg in segments:
        audio = synthesize(
            seg['text'],
            voice=seg.get('voice', 'en-US-AriaNeural'),
            tts_fn=tts_fn,
            rate=seg.get('rate', '+0%'),
            pitch=seg.get('pitch', '+0Hz'),
        )
        out.append(audio)
    return out


## Automated checks

In [ ]:

score, total = 0, 5
try:
    script = [
        {'text': 'Welcome to the show.', 'voice': 'en-US-AriaNeural'},
        {'text': 'Thanks for having me!', 'voice': 'en-US-GuyNeural', 'rate': '+10%'},
        {'text': "Let's dive in.",        'voice': 'en-US-AriaNeural'},
    ]
    parts = synthesize_segments(script, tts_fn=_mock_tts)

    # returns a list of correct length
    assert isinstance(parts, list) and len(parts) == 3
    score += 1; print("✅ returns list with correct item count")

    # each item is bytes
    assert all(isinstance(p, bytes) for p in parts)
    score += 1; print("✅ all items are bytes")

    # each item is non-empty
    assert all(len(p) > 0 for p in parts)
    score += 1; print("✅ all items are non-empty")

    # defaults work when voice/rate/pitch absent
    minimal = [{'text': 'No voice specified.'}]
    min_parts = synthesize_segments(minimal, tts_fn=_mock_tts)
    assert len(min_parts) == 1 and len(min_parts[0]) > 0
    score += 1; print("✅ segment with only text key works (defaults applied)")

    # different text → different audio
    assert parts[0] != parts[1], "Different text should give different bytes"
    score += 1; print("✅ different segments produce different audio bytes")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def synthesize_segments(segments, tts_fn=None):
    out = []
    for seg in segments:
        audio = synthesize(
            seg['text'],
            voice=seg.get('voice', 'en-US-AriaNeural'),
            tts_fn=tts_fn,
            rate=seg.get('rate', '+0%'),
            pitch=seg.get('pitch', '+0Hz'),
        )
        out.append(audio)
    return out
```

**Why not use a list comprehension?** A list comprehension would work, but the explicit loop is clearer for a multi-argument call. If an exception occurs, the explicit loop makes it easier to add error handling per segment in a production version.

</details>